# 01 — Clinical Trials Data Fetch
**Use case:** Cancer-wise Drug Intelligence  
**Source:** ClinicalTrials.gov v2 API  
**Query:** Broad oncology umbrella (all cancer types)  
**Output:** raw JSON · flat CSV · interventions CSV · locations CSV


In [1]:
import requests
import pandas as pd
import json
import os
import time
from tqdm.notebook import tqdm
from datetime import datetime

print("Imports OK")


Imports OK


## Configuration

In [2]:
# ── API ─────────────────────────────────────────────────────────────────────
BASE_URL  = "https://clinicaltrials.gov/api/v2/studies"
PAGE_SIZE = 1000   # max allowed

# ── QUERY ────────────────────────────────────────────────────────────────────
# Broad oncology umbrella — covers all cancer types
# ClinicalTrials.gov handles synonyms well but we use OR for maximum coverage
ONCOLOGY_QUERY = (
    "neoplasm OR cancer OR oncology OR tumor OR tumour OR "
    "carcinoma OR lymphoma OR leukemia OR sarcoma OR melanoma OR "
    "malignant OR malignancy"
)

# ── OUTPUT PATHS ─────────────────────────────────────────────────────────────
os.makedirs("data/raw",       exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

RAW_JSON_PATH      = "data/raw/oncology_trials_raw.json"
RAW_CSV_PATH       = "data/raw/oncology_trials_flat.csv"
INTERVENTIONS_PATH = "data/raw/oncology_interventions.csv"
LOCATIONS_PATH     = "data/raw/oncology_locations.csv"

print("Config set.")
print(f"Query : {ONCOLOGY_QUERY}")


Config set.
Query : neoplasm OR cancer OR oncology OR tumor OR tumour OR carcinoma OR lymphoma OR leukemia OR sarcoma OR melanoma OR malignant OR malignancy


## Step 1 — Fetch all oncology trials
No record limit — iterates all pages using `nextPageToken`.  
Estimated time: **20–40 minutes** depending on your connection.  
Do not close the notebook while this runs.


In [3]:
def fetch_all_oncology(query: str, page_size: int = 1000) -> list:
    params = {
        "query.cond" : query,
        "pageSize"   : page_size,
        "format"     : "json",
    }

    all_studies = []
    session     = requests.Session()
    max_retries = 5
    retry_delay = 10

    pbar = tqdm(desc="Fetching oncology trials", unit=" page")

    while True:
        for attempt in range(max_retries):
            try:
                resp = session.get(BASE_URL, params=params, timeout=60)
                resp.raise_for_status()
                data = resp.json()
                break
            except Exception as exc:
                if attempt < max_retries - 1:
                    pbar.write(f"Attempt {attempt+1} failed: {exc}. Retrying in {retry_delay}s...")
                    time.sleep(retry_delay)
                else:
                    pbar.close()
                    raise

        studies = data.get("studies", [])
        all_studies.extend(studies)
        pbar.update(1)
        pbar.set_postfix({"total_records": len(all_studies)})

        next_token = data.get("nextPageToken")
        if not next_token:
            break

        params["pageToken"] = next_token
        time.sleep(0.2)   # polite delay

    pbar.close()
    print(f"\nFetch complete — {len(all_studies):,} studies fetched.")
    return all_studies


studies = fetch_all_oncology(ONCOLOGY_QUERY)


Fetching oncology trials: 0 page [00:00, ? page/s]

Attempt 1 failed: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')). Retrying in 10s...

Fetch complete — 123,526 studies fetched.


## Step 2 — Save raw JSON
Complete unmodified objects. Source of truth.

In [4]:
print(f"Saving raw JSON...")
with open(RAW_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump({
        "metadata": {
            "query"        : ONCOLOGY_QUERY,
            "fetch_date"   : datetime.now().isoformat(),
            "total_records": len(studies),
        },
        "studies": studies
    }, f, ensure_ascii=False, indent=2)

size_mb = os.path.getsize(RAW_JSON_PATH) / (1024**2)
print(f"Saved {len(studies):,} studies ({size_mb:.1f} MB) → {RAW_JSON_PATH}")


Saving raw JSON...
Saved 123,526 studies (4661.0 MB) → data/raw/oncology_trials_raw.json


## Step 3 — Flatten to CSV
One row per trial. List fields stored as pipe-separated strings.  
Nothing is dropped here — column pruning happens in `02_eda_features.ipynb`.


In [5]:
def extract_fields(study: dict) -> dict:
    row      = {}
    protocol = study.get("protocolSection", {})

    # IDENTIFICATION
    ident = protocol.get("identificationModule", {})
    row["nct_id"]         = ident.get("nctId", "")
    row["brief_title"]    = ident.get("briefTitle", "")
    row["official_title"] = ident.get("officialTitle", "")

    # STATUS
    status = protocol.get("statusModule", {})
    row["overall_status"]          = status.get("overallStatus", "")
    row["status_verified_date"]    = status.get("statusVerifiedDate", "")
    row["start_date"]              = status.get("startDateStruct",          {}).get("date", "")
    row["primary_completion_date"] = status.get("primaryCompletionDateStruct", {}).get("date", "")
    row["completion_date"]         = status.get("completionDateStruct",     {}).get("date", "")
    row["first_posted_date"]       = status.get("studyFirstPostDateStruct", {}).get("date", "")
    row["last_update_posted_date"] = status.get("lastUpdatePostDateStruct", {}).get("date", "")
    expanded = status.get("expandedAccessInfo", {})
    row["expanded_access_available"] = expanded.get("hasExpandedAccess", False)

    # SPONSOR & COLLABORATORS
    sc = protocol.get("sponsorCollaboratorsModule", {})
    rp = sc.get("responsibleParty", {})
    row["responsible_party_type"]     = rp.get("type", "")
    row["responsible_party_inv_name"] = rp.get("investigatorFullName", "")
    row["responsible_party_affiliation"] = rp.get("investigatorAffiliation", "")
    ls = sc.get("leadSponsor", {})
    row["lead_sponsor_name"]  = ls.get("name",  "")
    row["lead_sponsor_class"] = ls.get("class", "")
    collabs = sc.get("collaborators", [])
    row["collaborators"]        = "|".join([c.get("name",  "") for c in collabs])
    row["collaborator_classes"] = "|".join([c.get("class", "") for c in collabs])
    row["num_collaborators"]    = len(collabs)

    # DESCRIPTION
    desc = protocol.get("descriptionModule", {})
    row["brief_summary"]        = desc.get("briefSummary",       "")
    row["detailed_description"] = desc.get("detailedDescription","")

    # CONDITIONS
    cond = protocol.get("conditionsModule", {})
    conditions_list   = cond.get("conditions", [])
    keywords_list     = cond.get("keywords",   [])
    row["conditions"]     = "|".join(conditions_list)
    row["keywords"]       = "|".join(keywords_list)
    row["num_conditions"] = len(conditions_list)

    # DESIGN
    design = protocol.get("designModule", {})
    row["study_type"] = design.get("studyType", "")
    phases = design.get("phases", [])
    row["phases"]     = "|".join(phases)
    row["phase_raw"]  = "|".join(phases) if phases else "NA"
    row["num_phases"] = len(phases)

    di = design.get("designInfo", {})
    row["allocation"]         = di.get("allocation",        "")
    row["intervention_model"] = di.get("interventionModel", "")
    row["primary_purpose"]    = di.get("primaryPurpose",    "")
    row["observational_model"]= di.get("observationalModel","")
    mi = di.get("maskingInfo", {})
    row["masking"]    = mi.get("masking", "")
    row["who_masked"] = "|".join(mi.get("whoMasked", []))

    enr = design.get("enrollmentInfo", {})
    row["enrollment_count"] = enr.get("count", None)
    row["enrollment_type"]  = enr.get("type",  "")

    # ARMS & INTERVENTIONS
    ai   = protocol.get("armsInterventionsModule", {})
    arms = ai.get("armGroups", [])
    row["num_arms"]   = len(arms)
    row["arm_labels"] = "|".join([a.get("label", "") for a in arms])
    row["arm_types"]  = "|".join([a.get("type",  "") for a in arms])

    interventions = ai.get("interventions", [])
    row["num_interventions"]  = len(interventions)
    row["intervention_names"] = "|".join([i.get("name", "") for i in interventions])
    row["intervention_types"] = "|".join([i.get("type", "") for i in interventions])

    drugs = [i for i in interventions if i.get("type","").upper() == "DRUG"]
    row["drug_names"]       = "|".join([d.get("name", "") for d in drugs])
    row["drug_other_names"] = "|".join([
        "|".join(d.get("otherNames", [])) for d in drugs
    ])
    row["num_drugs"] = len(drugs)

    bios = [i for i in interventions
            if i.get("type","").upper() in ("BIOLOGICAL","GENETIC")]
    row["biological_names"] = "|".join([b.get("name", "") for b in bios])
    row["num_biologicals"]  = len(bios)

    # OUTCOMES
    outcomes  = protocol.get("outcomesModule", {})
    primary   = outcomes.get("primaryOutcomes",   [])
    secondary = outcomes.get("secondaryOutcomes", [])
    row["num_primary_outcomes"]       = len(primary)
    row["primary_outcome_measures"]   = "|".join([o.get("measure",   "") for o in primary])
    row["primary_outcome_timeframes"] = "|".join([o.get("timeFrame", "") for o in primary])
    row["num_secondary_outcomes"]     = len(secondary)
    row["secondary_outcome_measures"] = "|".join([o.get("measure",   "") for o in secondary])

    # ELIGIBILITY
    elig = protocol.get("eligibilityModule", {})
    row["eligibility_criteria"] = elig.get("eligibilityCriteria", "")
    row["healthy_volunteers"]   = elig.get("healthyVolunteers", False)
    row["sex"]                  = elig.get("sex", "")
    row["minimum_age"]          = elig.get("minimumAge", "")
    row["maximum_age"]          = elig.get("maximumAge", "")
    row["std_ages"]             = "|".join(elig.get("stdAges", []))

    # CONTACTS & LOCATIONS
    cl   = protocol.get("contactsLocationsModule", {})
    locs = cl.get("locations", [])
    row["num_locations"] = len(locs)
    countries = sorted(set(
        loc.get("country","") for loc in locs if loc.get("country")
    ))
    row["countries"]     = "|".join(countries)
    row["num_countries"] = len(countries)

    officials = cl.get("overallOfficials", [])
    pis = [o for o in officials
           if "PRINCIPAL_INVESTIGATOR" in o.get("role","").upper()]
    row["num_pis"]         = len(pis)
    row["pi_names"]        = "|".join([o.get("name",        "") for o in pis])
    row["pi_affiliations"] = "|".join([o.get("affiliation", "") for o in pis])

    # OVERSIGHT
    ov = protocol.get("oversightModule", {})
    row["has_dmc"]            = ov.get("oversightHasDmc",      False)
    row["fda_regulated_drug"] = ov.get("isFdaRegulatedDrug",   False)

    # IPD SHARING
    ipd = protocol.get("ipdSharingStatementModule", {})
    row["ipd_sharing"] = ipd.get("ipdSharing", "")

    # HAS RESULTS
    row["has_results"] = study.get("hasResults", False)

    return row


print("extract_fields() defined.")


extract_fields() defined.


In [6]:
print(f"Flattening {len(studies):,} studies...")
rows   = []
errors = []

for study in tqdm(studies, desc="Flattening"):
    try:
        rows.append(extract_fields(study))
    except Exception as exc:
        nct_id = (study.get("protocolSection", {})
                       .get("identificationModule", {})
                       .get("nctId", "UNKNOWN"))
        errors.append({"nct_id": nct_id, "error": str(exc)})

df = pd.DataFrame(rows)

if errors:
    print(f"\nWarning: {len(errors)} studies could not be processed.")

df.to_csv(RAW_CSV_PATH, index=False, encoding="utf-8-sig")
size_mb = os.path.getsize(RAW_CSV_PATH) / (1024**2)
print(f"\nFlat CSV saved ({size_mb:.1f} MB) → {RAW_CSV_PATH}")
print(f"Shape : {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(3)


Flattening 123,526 studies...


Flattening:   0%|          | 0/123526 [00:00<?, ?it/s]


Flat CSV saved (640.7 MB) → data/raw/oncology_trials_flat.csv
Shape : 123,526 rows × 68 columns


,nct_id,brief_title,official_title,overall_status,status_verified_date,start_date,primary_completion_date,completion_date,first_posted_date,last_update_posted_date,...,num_locations,countries,num_countries,num_pis,pi_names,pi_affiliations,has_dmc,fda_regulated_drug,ipd_sharing,has_results
0,NCT07491497,A Phase 1/2 Study of TRI-611 in ALK-Positive N...,"A Phase 1/2, Dose Escalation and Expansion Stu...",RECRUITING,2026-05,2026-03-11,2029-05-31,2034-01-30,2026-03-24,2026-05-07,...,7,United States,1,0,,,False,True,,False
1,NCT00669097,"Absorption, Distribution, Metabolism and Excre...","An Open-Label, Non-Randomized, Single-Center, ...",COMPLETED,2011-08,2008-04,2010-01,,2008-04-29,2020-12-21,...,1,Netherlands,1,0,,,False,False,,False
2,NCT07490990,"A Real-World, Single-Arm Study Protocol of Dis...","A Real-World, Single-Arm Study Protocol of Dis...",NOT_YET_RECRUITING,2026-03,2026-06,2028-06,2028-12,2026-03-24,2026-03-24,...,0,,0,0,,,False,False,,False


## Step 4 — Interventions table
One row per drug per trial. Key for drug intelligence use case.

In [7]:
def extract_interventions(studies: list) -> pd.DataFrame:
    rows = []
    for study in studies:
        proto  = study.get("protocolSection", {})
        nct_id = proto.get("identificationModule", {}).get("nctId", "")
        phases = proto.get("designModule", {}).get("phases", [])
        status = proto.get("statusModule", {}).get("overallStatus", "")
        sponsor_class = (proto.get("sponsorCollaboratorsModule", {})
                              .get("leadSponsor", {}).get("class", ""))
        conditions = proto.get("conditionsModule", {}).get("conditions", [])

        for inv in (proto.get("armsInterventionsModule", {})
                        .get("interventions", [])):
            rows.append({
                "nct_id"           : nct_id,
                "overall_status"   : status,
                "phases"           : "|".join(phases),
                "sponsor_class"    : sponsor_class,
                "conditions"       : "|".join(conditions),
                "intervention_type": inv.get("type",  ""),
                "intervention_name": inv.get("name",  ""),
                "other_names"      : "|".join(inv.get("otherNames", [])),
                "description"      : inv.get("description", "")[:500],
                "arm_group_labels" : "|".join(inv.get("armGroupLabels", [])),
            })
    return pd.DataFrame(rows)


df_inv = extract_interventions(studies)
df_inv.to_csv(INTERVENTIONS_PATH, index=False, encoding="utf-8-sig")
print(f"Interventions table : {df_inv.shape[0]:,} rows × {df_inv.shape[1]} cols → {INTERVENTIONS_PATH}")
df_inv.head(3)


Interventions table : 245,442 rows × 10 cols → data/raw/oncology_interventions.csv


,nct_id,overall_status,phases,sponsor_class,conditions,intervention_type,intervention_name,other_names,description,arm_group_labels
0,NCT07491497,RECRUITING,PHASE1|PHASE2,INDUSTRY,ALK-positive NSCLC|ALK-Positive Lung Cancer|AL...,DRUG,TRI-611,,oral ALK molecular glue degrader,Part 1: Dose Escalation and Backfill|Part 2: C...
1,NCT00669097,COMPLETED,PHASE1,INDUSTRY,Advanced Solid Malignancies,DRUG,TKI258,,,TKI258
2,NCT07490990,NOT_YET_RECRUITING,,OTHER,HER2-positive Advanced Gastric Cancer or Gastr...,DRUG,Disitamab vedotin + Sintilimab + Multimodality...,,Disitamab vedotin (generic name; code: RC48; C...,


## Step 5 — Locations table
One row per site per trial. Key for geography analysis.

In [8]:
def extract_locations(studies: list) -> pd.DataFrame:
    rows = []
    for study in studies:
        proto  = study.get("protocolSection", {})
        nct_id = proto.get("identificationModule", {}).get("nctId", "")
        phases = proto.get("designModule", {}).get("phases", [])
        status = proto.get("statusModule", {}).get("overallStatus", "")

        for loc in (proto.get("contactsLocationsModule", {})
                        .get("locations", [])):
            rows.append({
                "nct_id"         : nct_id,
                "overall_status" : status,
                "phases"         : "|".join(phases),
                "facility"       : loc.get("facility", ""),
                "city"           : loc.get("city",     ""),
                "state"          : loc.get("state",    ""),
                "country"        : loc.get("country",  ""),
                "zip"            : loc.get("zip",      ""),
                "location_status": loc.get("status",   ""),
                "geo_lat"        : loc.get("geoPoint", {}).get("lat", None),
                "geo_lon"        : loc.get("geoPoint", {}).get("lon", None),
            })
    return pd.DataFrame(rows)


df_loc = extract_locations(studies)
df_loc.to_csv(LOCATIONS_PATH, index=False, encoding="utf-8-sig")
print(f"Locations table : {df_loc.shape[0]:,} rows × {df_loc.shape[1]} cols → {LOCATIONS_PATH}")
df_loc.head(3)


Locations table : 1,248,061 rows × 11 cols → data/raw/oncology_locations.csv


,nct_id,overall_status,phases,facility,city,state,country,zip,location_status,geo_lat,geo_lon
0,NCT07491497,RECRUITING,PHASE1|PHASE2,University of Colorado Cancer Center,Aurora,Colorado,United States,80045,RECRUITING,39.72943,-104.83192
1,NCT07491497,RECRUITING,PHASE1|PHASE2,Washington University Medical Center,St Louis,Missouri,United States,63130,RECRUITING,38.62727,-90.19789
2,NCT07491497,RECRUITING,PHASE1|PHASE2,Memorial Sloan-Kettering Cancer Center,New York,New York,United States,10065,RECRUITING,40.71427,-74.00597


## Summary

In [9]:
div = "=" * 60
print(div)
print("FETCH SUMMARY — ONCOLOGY TRIALS")
print(div)
print(f"Total trials            : {len(df):,}")
print(f"Total columns           : {len(df.columns)}")
print()
print("Phase distribution:")
print(df["phase_raw"].value_counts().head(15).to_string())
print()
print("Overall status:")
print(df["overall_status"].value_counts().head(10).to_string())
print()
print("Lead sponsor class:")
print(df["lead_sponsor_class"].value_counts().to_string())
print()
print("Study type:")
print(df["study_type"].value_counts().to_string())
print()
print(f"Trials with enrollment  : {df['enrollment_count'].notna().sum():,}")
print(f"Total targeted patients : {df['enrollment_count'].sum():,.0f}")
print()
print("Files saved:")
for p in [RAW_JSON_PATH, RAW_CSV_PATH, INTERVENTIONS_PATH, LOCATIONS_PATH]:
    mb = os.path.getsize(p) / (1024**2)
    print(f"  {mb:6.1f} MB  →  {p}")
print(div)
print("Next → 02_eda_features.ipynb")


FETCH SUMMARY — ONCOLOGY TRIALS
Total trials            : 123,526
Total columns           : 68

Phase distribution:
phase_raw
NA               52467
PHASE2           30208
PHASE1           16598
PHASE3            9618
PHASE1|PHASE2     8571
PHASE4            2635
EARLY_PHASE1      1942
PHASE2|PHASE3     1487

Overall status:
overall_status
COMPLETED                  52933
UNKNOWN                    19993
RECRUITING                 19503
TERMINATED                 11119
ACTIVE_NOT_RECRUITING       7911
NOT_YET_RECRUITING          6304
WITHDRAWN                   4101
ENROLLING_BY_INVITATION      743
SUSPENDED                    483
NO_LONGER_AVAILABLE          228

Lead sponsor class:
lead_sponsor_class
OTHER        86389
INDUSTRY     27550
NIH           3942
NETWORK       2707
OTHER_GOV     2575
FED            269
INDIV           57
UNKNOWN         37

Study type:
study_type
INTERVENTIONAL     97396
OBSERVATIONAL      25694
EXPANDED_ACCESS      436

Trials with enrollment  : 121,310
To

In [10]:
# Quick column exploration
print(df.columns.tolist())
print("\n")
print(df.dtypes)
print("\n")
print(df.isnull().sum())


['nct_id', 'brief_title', 'official_title', 'overall_status', 'status_verified_date', 'start_date', 'primary_completion_date', 'completion_date', 'first_posted_date', 'last_update_posted_date', 'expanded_access_available', 'responsible_party_type', 'responsible_party_inv_name', 'responsible_party_affiliation', 'lead_sponsor_name', 'lead_sponsor_class', 'collaborators', 'collaborator_classes', 'num_collaborators', 'brief_summary', 'detailed_description', 'conditions', 'keywords', 'num_conditions', 'study_type', 'phases', 'phase_raw', 'num_phases', 'allocation', 'intervention_model', 'primary_purpose', 'observational_model', 'masking', 'who_masked', 'enrollment_count', 'enrollment_type', 'num_arms', 'arm_labels', 'arm_types', 'num_interventions', 'intervention_names', 'intervention_types', 'drug_names', 'drug_other_names', 'num_drugs', 'biological_names', 'num_biologicals', 'num_primary_outcomes', 'primary_outcome_measures', 'primary_outcome_timeframes', 'num_secondary_outcomes', 'second

In [11]:
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\nIf empty above = genuinely no nulls")
print(df[['enrollment_count', 'drug_names', 'conditions', 'phases']].isnull().sum())

enrollment_count    2216
dtype: int64

If empty above = genuinely no nulls
enrollment_count    2216
drug_names             0
conditions             0
phases                 0
dtype: int64


In [12]:
print(df['enrollment_count'].describe())
print("\nTop 10 highest enrollment:")
print(df[['nct_id','brief_title','enrollment_count','study_type','phase_raw']]
      .sort_values('enrollment_count', ascending=False)
      .head(10).to_string())

count    1.213100e+05
mean     3.847710e+03
std      3.347637e+05
min      0.000000e+00
25%      2.600000e+01
50%      6.000000e+01
75%      1.740000e+02
max      1.000000e+08
Name: enrollment_count, dtype: float64

Top 10 highest enrollment:
             nct_id                                                                                                               brief_title  enrollment_count     study_type phase_raw
76089   NCT01166009                                                                                                  CIBMTR Research Database        99999999.0  OBSERVATIONAL        NA
70098   NCT06268535  Identification of Anticancer Drugs Associated With Cancer Therapy-related Cardiac Dysfunction: a Pharmacovigilance Study        36580288.0  OBSERVATIONAL        NA
110689  NCT00904579                                                    Cancer Risk in Organ Transplant Recipients and End-Stage Renal Disease        19929901.0  OBSERVATIONAL        NA
55727   NCT025531

In [13]:
print(df[['nct_id','drug_names','drug_other_names','phase_raw','conditions']]
      .dropna(subset=['drug_names'])
      .head(10)
      .to_string())

        nct_id                                                                                                                                                                                              drug_names                                                                                                                                                                                                                                                                                                                                                                                            drug_other_names      phase_raw                                                                                                                                                                                                               conditions
0  NCT07491497                                                                                                                                               

In [14]:
print(df[df['phase_raw']=='NA']['study_type'].value_counts())

study_type
INTERVENTIONAL     26337
OBSERVATIONAL      25694
EXPANDED_ACCESS      436
Name: count, dtype: int64


In [15]:
empty_drugs = df[df['drug_names']=='']
print(empty_drugs['study_type'].value_counts())
print("\n")
print(empty_drugs['phase_raw'].value_counts().head(10))

study_type
INTERVENTIONAL     34773
OBSERVATIONAL      23310
EXPANDED_ACCESS       85
Name: count, dtype: int64


phase_raw
NA               46686
PHASE2            3634
PHASE1            3057
PHASE3            1767
PHASE1|PHASE2     1654
EARLY_PHASE1       600
PHASE4             479
PHASE2|PHASE3      291
Name: count, dtype: int64


In [16]:
import re
combo = df[df['drug_names'].str.contains(r'\+', na=False)]
print(f"Trials with + in drug names: {len(combo)}")
print(combo['drug_names'].head(10).to_string())

Trials with + in drug names: 2181
2      Disitamab vedotin + Sintilimab + Multimodality...
49     Tamoxifen or Aromatase Inhibitor or Aromatase ...
143    Metformin + Myocet + Cyclophosphamide|Myocet +...
219    Paclitaxel/Bev (control)|Paclitaxel/Bev + ZA (...
423    FOLFOXIRI + QL1706 + Bevacizumab|FOLFOX+QL1706...
452    IMM2510|Chemotherapy (pemetrexed + cisplatin/c...
455                            capecitabine + ganetespib
463               Chemotherapy + vitamin D2|Chemotherapy
468    Surgery +(Cisplatin 80 mg/㎡+5FU 800mg/㎡×5days)...
514    Group A (FIT): Avapritinib + IA regimen|Group ...


In [17]:
print(df['phase_raw'].value_counts())
print("\n")
print(df['phases'].value_counts().head(20))

phase_raw
NA               52467
PHASE2           30208
PHASE1           16598
PHASE3            9618
PHASE1|PHASE2     8571
PHASE4            2635
EARLY_PHASE1      1942
PHASE2|PHASE3     1487
Name: count, dtype: int64


phases
PHASE2           30208
NA               26330
                 26137
PHASE1           16598
PHASE3            9618
PHASE1|PHASE2     8571
PHASE4            2635
EARLY_PHASE1      1942
PHASE2|PHASE3     1487
Name: count, dtype: int64


In [18]:
print(df[['nct_id','conditions','phase_raw','study_type']].head(20).to_string())

         nct_id                                                                                                                                                                                                               conditions      phase_raw      study_type
0   NCT07491497                                                                                                                                      ALK-positive NSCLC|ALK-Positive Lung Cancer|ALK-positive Non-small Cell Lung Cancer  PHASE1|PHASE2  INTERVENTIONAL
1   NCT00669097                                                                                                                                                                                              Advanced Solid Malignancies         PHASE1  INTERVENTIONAL
2   NCT07490990                                                                                                                                        HER2-positive Advanced Gastric Cancer or Gastroesophageal